In [2]:
import importlib
import torch.nn as nn


import sys
import json
import random
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset, Image as HFImage

In [3]:
sys.path.append("..")
from src.data.preprocessing import Ai4MarsSegmentation, IMG_SIZE, CLASS_NAMES, IGNORE_INDEX

SEED = 42
BATCH_SIZE = 8
NUM_WORKERS = 0     # keep 0 while debugging; raise later for speed


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)
print("Image size:", IMG_SIZE, "| Batch size:", BATCH_SIZE)

Device: mps
Image size: 512 | Batch size: 8


In [4]:
# Load dataset
IMAGE_COLUMNS = ["image", "label_mask", "rover_mask", "range_mask"]

dataset = load_dataset("hassanjbara/AI4MARS")

raw_datasets = {}
for split, data in dataset.items():
    for column in IMAGE_COLUMNS:
        data = data.cast_column(column, HFImage(decode=False))
    raw_datasets[split] = data

for k, v in raw_datasets.items():
    print(k, len(v))

train 18130
test_min1 322
test_min2 322
test_min3 322


In [5]:
# Datasets and loaders from the saved splits

with open("../data/splits.json") as f:
    splits = json.load(f)                   # Reads the file and converts JSON into a Python dictionary.

train_ds = Ai4MarsSegmentation(raw_datasets["train"], splits["train"])
val_ds   = Ai4MarsSegmentation(raw_datasets["train"], splits["val"])
test_ds  = {                                # Dictionary Comprehension
    s: Ai4MarsSegmentation(raw_datasets[s], range(len(raw_datasets[s])))
    for s in ["test_min1", "test_min2", "test_min3"]
}

pin = device.type == "cuda"
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=pin, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)
test_loaders = {
    s: DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                  num_workers=NUM_WORKERS, pin_memory=pin)
    for s, ds in test_ds.items()
}

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: { {s: len(d) for s, d in test_ds.items()} }")

Train: 14312 | Val: 1589 | Test: {'test_min1': 322, 'test_min2': 322, 'test_min3': 322}


             Hugging Face Dataset
                      |
                      v
               raw_datasets
                      |
          +-----------+-----------+
          |           |           |
          v           v           v
       Train         Val         Test
          |           |           |
          v           v           v
      train_ds      val_ds      test_ds
          |           |           |
          v           v           v
    train_loader  val_loader  test_loaders
          |           |           |
          +-----------+-----------+
                      |
                      v
                Model Training
                & Evaluation

In [6]:
# Metrics unit test 
from src.evaluation.metrics import ConfusionMatrix, format_report

targets = torch.tensor([0, 0, 1, 1, 2, 3, 255, 255])
preds   = torch.tensor([0, 1, 1, 1, 2, 0,   0,   3])   # last two are ignored

cm = ConfusionMatrix()
cm.update(preds, targets)
res = cm.compute()

assert res["pixels_evaluated"] == 6                      # 255 pixels skipped
assert abs(res["iou"][0] - 1/3) < 1e-9                   # tp=1, gt=2, pred=2 -> 1/3
assert abs(res["iou"][1] - 2/3) < 1e-9                   # tp=2, gt=2, pred=3 -> 2/3
assert abs(res["iou"][2] - 1.0) < 1e-9
assert res["iou"][3] == 0.0                              # class 3 never predicted correctly
assert abs(res["pixel_acc"] - 4/6) < 1e-9
print(format_report(res))
print("Metrics tests passed.")

mIoU: 0.5000 | pixel acc: 0.6667
  Soil      IoU 0.3333 | recall 0.5000
  Bedrock   IoU 0.6667 | recall 1.0000
  Sand      IoU 1.0000 | recall 1.0000
  Big Rock  IoU 0.0000 | recall 0.0000
Metrics tests passed.


              Model Predictions
                      |
                      v
               Ground Truth
                      |
                      v
                ConfusionMatrix
                      |
                      v
                   update()
                      |
                      v
            Accumulate Pixel Counts
                      |
                      v
                  compute()
                      |
          +-----------+-----------+-----------+
          |           |           |           |
          v           v           v           v
         IoU        Recall    Pixel Acc.     mIoU
          |           |           |           |
          +-----------+-----------+-----------+
                      |
                      v
                format_report()
                      |
                      v
              Evaluation Report

In [7]:
# Model tests
import torch.nn as nn
from src.models.unet import UNet, count_parameters

model = UNet(in_channels=3, num_classes=4, base=32).to(device)
print(f"Parameters: {count_parameters(model) / 1e6:.2f} M")

images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

model.train()
logits = model(images)
assert logits.shape == (images.shape[0], 4, IMG_SIZE, IMG_SIZE), logits.shape

criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
loss = criterion(logits, labels)
loss.backward()

print("Logits:", tuple(logits.shape), "| Loss:", round(loss.item(), 4))
print("Expected initial loss is about ln(4) =", round(float(np.log(4)), 3))
assert torch.isfinite(loss)
print("Model tests passed.")

Parameters: 7.76 M
Logits: (8, 4, 512, 512) | Loss: 1.4792
Expected initial loss is about ln(4) = 1.386
Model tests passed.


             Input Image
                   |
                   v
                  enc1 -------- e1 ------------------+
                   |                                 |
                  Pool                               |
                   |                                 |
                  enc2 -------- e2 --------------+   |
                   |                             |   |
                  Pool                           |   |
                   |                             |   |
                  enc3 -------- e3 ----------+   |   |
                   |                         |   |   |
                  Pool                       |   |   |
                   |                         |   |   |
                  enc4 -------- e4 ------+   |   |   |
                   |                     |   |   |   |
                  Pool                   |   |   |   |
                   |                     |   |   |   |
              Bottleneck                 |   |   |   |
                   |                     |   |   |   |
                  up4                    |   |   |   |
                   |                     |   |   |   |
                Concat <-----------------+   |   |   |
                   |                         |   |   |
                  dec4                       |   |   |
                   |                         |   |   |
                  up3                        |   |   |
                   |                         |   |   |
                Concat <---------------------+   |   |
                   |                             |   |
                  dec3                           |   |
                   |                             |   |
                  up2                            |   |
                   |                             |   |
                Concat <-------------------------+   |
                   |                                 |
                  dec2                               |
                   |                                 |
                  up1                                |
                   |                                 |
                Concat <-----------------------------+
                   |
                  dec1
                   |
                   v
             Segmentation Head
                   |
                   v
             Output (B, 4, H, W)